# Task Statement 1.7: Manage Session State, Resumption, and Forking

> Domain 1 (27% of the exam) · Scenarios 2 (Code Generation) and 4 (Developer Productivity) · [Back to Domain 1](../README.md)

## In One Minute

- **What**: a session is a saved conversation. You can continue it, branch it, or replace it with a summary.
- **Resume**: `claude --resume <session-name>` (or `-r`) continues a named session; name one with `--name` / `-n`. In the Agent SDK (the Python/TypeScript code library), pass `resume=<session_id>`. Good when nothing important changed.
- **Fork**: `fork_session` (CLI terminal: `--fork-session` with `--resume`; Python SDK: `fork_session=True`; TypeScript: `forkSession: true`) makes an independent branch from a shared baseline. A fork copies the conversation history, not the files on disk. Good for comparing approaches.
- **Stale results**: if files changed, tell the resumed session which ones. If the old tool results are mostly stale, start a **new session with a structured summary**.
- **Exam answer in one line**: resume when results are still valid, fork to compare, summarise into a fresh session when tool results are stale.

---

## 🧒 Explain Like I'm 13

**The picture:** A video game save file. **Resume** when the world is unchanged. If a friend moved the furniture, **tell the game what changed** before you continue. If the whole map changed, **start a new game** and bring a notes page. **Fork** means copying the save to try two different paths.

**In plain steps:**
1. Nothing changed: resume (`--resume <name>`).
2. A few files changed: resume and list them so Claude re-reads them.
3. Lots changed: new session, seeded with a short summary of verified facts.
4. Two options to compare: fork twice from the same save.

**Tiny example:** Claude read `auth.py` yesterday. Today it was rewritten. If you resume silently, Claude edits the old lines from memory. Say "auth.py changed, re-read it" and it is fine.

---

## 📌 Overview

### Why it matters
Work spans days. On Monday Claude read `db/conn.py` and saw a pool size of 5. On Tuesday you edited it. If you resume Monday's session and say nothing, Claude still believes the old file. It will suggest changes to code that no longer exists. The old tool results are **stale**.

---

## 📖 Concepts Explained

### 1. The Context Stale-State Problem
Large engineering workflows do not happen in a single, instantaneous API turn. Real-world tasks span days, multiple git branches, and team collaboration.
Consider this failure sequence:
1. On Monday, Claude Code explores a codebase and reads `db/connection.py`, noting that the database connection pool size is 5.
2. On Tuesday, a team engineer refactors the database layer, migrating from synchronous SQLAlchemy to async asyncpg, renaming connection parameters, and moving pooling logic into `config/database.py`.
3. On Wednesday, you return to your terminal and resume Monday's session.
4. If you simply resume without notifying the session, **Claude still believes `db/connection.py` exists in its old state**. When instructed to *"add connection pooling retry logic"*, Claude emits surgical `Edit` calls targeting lines and symbols that no longer exist on disk, producing corrupted patches, hallucinated imports, or catastrophic build failures.

In conversational agents, **past tool outputs in conversation history are frozen snapshots of time**. When reality moves, those snapshots become toxic misinformation.

### 2. The Three State Management Strategies Compared

| Strategy | Mechanics | Best Used When | Architectural Trade-Off |
|---|---|---|---|
| **Deterministic Resume (`--resume <name>`)** | Restores the exact prior conversation history, system prompt, and tool results without alteration. | Short breaks (hours/overnight) where the filesystem and external state are **identical**. | Fast and zero-context-overhead, but fatal if underlying files changed. |
| **Differential Resume with Changed-Files Notice** | Resumes the conversation history but immediately injects a synthetic user turn listing specific modified files. | A small, localized set of files (1–3 files) were touched. | Preserves architectural reasoning while explicitly instructing Claude to invalidate cached file reads. |
| **Session Forking (`fork_session`)** | Deep-copies the existing conversation history into two independent branches (Branch A and Branch B) sharing an identical lineage root. | Exploring alternative architectural implementations (e.g., comparing Prisma vs Asyncpg) from a common analysis baseline. | Prevents cross-contamination between experimental branches. Changes in Fork A never leak into Fork B. |
| **Fresh Session with Structured Summary Manifest** | Discards raw transcripts and stale tool outputs entirely. Starts a pristine session initialized with an immutable Markdown summary of verified decisions. | Large upstream git merges, significant time gaps, or major refactors where 80%+ of prior tool outputs are stale. | Eliminates token bloat and stale hallucinations; forces human/architect to explicitly curate the state of truth. |

### 3. Deep-Dive: When to Fork vs. When to Summarize
A classic exam dilemma tests the boundary between `fork_session` and a fresh session with an injected summary:
- **Choose `fork_session`** when you have completed a time-consuming baseline analysis (e.g., spending 30 turns mapping AST dependencies) and now need to implement two competing approaches to compare benchmarks. The baseline analysis remains 100% valid for both branches.
- **Choose Fresh Session with Structured Summary** when the working tree has drifted. Resuming a 50-turn conversation when 40 upstream commits were merged floods the context with thousands of tokens of obsolete code snippets. The correct pattern is to extract verified architectural decisions into a Markdown summary:
  ```markdown
  # Architecture Migration State Manifest
  - **Decision**: Migrate to asyncpg connection pool.
  - **Confirmed Facts**: 45 legacy sync models identified in `models/legacy/`.
  - **Modifications**: `db/connection.py` deleted; pool config moved to `config/database.py`.
  - **Current Directive**: Implement async health-check endpoint.
  ```
  Starting a fresh session with this concise summary cuts context usage by 85% while guaranteeing zero hallucinated legacy dependencies.

### 4. Implementation in This Repository
In [main.py](main.py), the `SessionManager` class implements these exact state transitions:
- `SessionManager.resume(name, changed_files=[...])` inspects the delta list. If files changed, it automatically appends a targeted invalidation prompt instructing the agent to re-read those specific files before executing edits.
- `SessionState.fork(new_name)` performs a deep copy of the message history and links the child session to the parent session UUID.
- `SessionManager.resume_with_injected_summary(name, summary, new_instruction)` purges obsolete conversation turns, seeding the new context with the verified summary and current directive.

---

## 🧪 Worked Example

### ❌ Invalid Example (Anti-Pattern / What Breaks)
*(Anti-Pattern: Blind Resumption After Major Rebase)*
```bash
# BROKEN: Resuming a 50-turn session after 40 commits were rebased out-of-band
claude --resume legacy_db_analysis "Now implement the new connection pool"

# Why it fails in production:
# 1. Stale tool cache: Claude's context still contains 20,000 tokens of obsolete code from db/conn.py.
# 2. Conflicted edits: Claude generates diffs targeting deleted functions, failing with edit collisions.
# 3. Context bloat: 80% of prior turns describe architecture that was rewritten or abandoned.
```

### ✅ Valid Example (Production Pattern)
*(Production Pattern: Fork for Parallel Branches & Manifest Ingestion After Drifts)*
```bash
# SCENARIO A: Comparing two competing ORMs from a shared baseline analysis
# Fork session twice so exploration in Fork A never contaminates Fork B:
claude --resume legacy_db_analysis --fork-session "Prototype using Async SQLAlchemy"
claude --resume legacy_db_analysis --fork-session "Prototype using Prisma ORM"

# SCENARIO B: After major upstream git rebase (40 commits merged)
# Discard stale transcripts; start a clean session initialized with a verified state manifest:
claude "Migrate connection pool" << 'EOF'
# Migration State Manifest
- Decision: Migrate from sync psycopg2 to asyncpg.
- Confirmed: 45 legacy sync models identified in models/legacy/.
- File changes: db/conn.py was replaced with config/database.py in commit 8f92a1.
- Current Directive: Implement async connection health-check endpoint.
EOF
```

**Timeline walkthrough:**
Monday: you analyze a legacy database layer in session `legacy_db_analysis`. Claude reads `db/conn.py` (pool size 5).

| Day | Event | Right move |
|---|---|---|
| Monday | Baseline analysis is done | Keep session `legacy_db_analysis` |
| Monday | You want to compare Async SQLAlchemy and Prisma | **Fork** twice. Each branch gets its own question. The parent is untouched |
| Tuesday | You edit `db/conn.py` and `db/pool.py` | `--resume legacy_db_analysis`, then tell it: "These files changed: `db/conn.py`, `db/pool.py`. Re-read them." |
| Next week | You merged forty upstream commits | **New session** with a summary: "Decision: Async SQLAlchemy 2.0. 50 sync models. Changed since: schema, pool config." Then the new task: "Produce the migration plan." |

Each choice matches how much of the old context is still true.

---

## 🔑 Key Terms

- **Conversational Session State**: The persisted multi-turn transcript, tool outputs, and system configurations stored across user interactions.
- **Deterministic Session Resumption (`--resume <name>`)**: Restoring an existing conversation session verbatim, ideal when external repository files and environment state remain 100% identical.
- **Differential Resumption with Changed-Files Notice**: Continuing a conversation session while explicitly injecting a synthetic turn listing modified files, instructing the model to invalidate cached tool outputs.
- **Session Lineage Forking (`fork_session`)**: Deep-copying an existing conversation lineage into an isolated child session to explore alternative prototypes without polluting the parent baseline.
- **Stale Tool Context Degradation**: The operational failure where Claude acts on obsolete code or database outputs cached in earlier turns after real-world files were altered out-of-band.
- **Structured State Manifest Seeding**: Purging obsolete conversational transcripts entirely after large rebases and initializing a fresh session with an immutable Markdown summary of verified decisions.

---

## 🏗️ Architecture

```mermaid
---
config:
  htmlLabels: false
---
flowchart TD
    Base["Baseline session<br>(analysis done)"] --> Choice{"What changed?"}

    Choice -->|"Compare two approaches"| Fork["fork_session"]
    Fork --> A["Branch A<br>(isolated)"]
    Fork --> B["Branch B<br>(isolated)"]

    Choice -->|"Nothing changed"| Resume["--resume session-name"]
    Choice -->|"A few files changed"| ResumeInform["--resume session-name<br>and list changed files"]

    Choice -->|"Many changes<br>(tool results stale)"| Fresh["New session with<br>structured summary"]
    Fresh --> Clean["Only verified facts<br>and the new task"]
```

---

## ⚠️ Exam Traps and Correct Patterns

### Trap 1: Resuming with stale tool results and saying nothing

**Exam scenario**: Yesterday Claude read `auth.py` (lines 40-60 held the login check). Overnight a teammate rewrote it. You resume and ask "add rate limiting to the login check".

```text
ANTI-PATTERN: resume, then prompt as if nothing changed.
  Claude trusts its old Read result and edits lines 40-60 that no longer exist.

PATTERN (small change): resume and say what changed.
  "Since we last spoke, auth.py and session.py were rewritten. Re-read them before editing."

PATTERN (large change): start a fresh session seeded with a structured summary.
```

| Change since last session | Choice |
|---|---|
| Nothing | Resume |
| A few files | Resume and name the changed files |
| Many files, or old tool output is mostly wrong | Fresh session with a summary |

### Trap 2: Exploring two designs in one thread

**Exam scenario**: Compare a Redis cache against an in-memory cache for the same service.

In one thread, the Redis discussion becomes background for the in-memory one. Conclusions leak both ways, so neither option is evaluated fairly. Use a **fork** from the shared baseline.

```bash
claude --resume cache-analysis --fork-session   # branch A: Redis
claude --resume cache-analysis --fork-session   # branch B: in-memory
```

SDK: `resume=session_id` with `fork_session=True` creates a new session id that starts from the same history.

### Trap 3: A vague summary for a fresh session

```text
ANTI-PATTERN: "We looked at the database and found some issues."

PATTERN:
  Verified facts: orders table has no index on customer_id (confirmed with EXPLAIN, 2026-10-28)
  Decisions: add composite index (customer_id, created_at); no schema change
  Changed since: migrations/0042 added; orders.py query rewritten
  Next task: benchmark the new index on staging
  Do NOT reuse: earlier slow-query timings (measured before the change)
```

Put in verified decisions and facts. Leave out raw old tool output, which is the thing you are trying to avoid.

### Trap 4: Pasting the whole transcript instead of resuming

Pasting a long transcript into a new prompt costs tokens, loses the structured turns and `tool_use` pairing, and still carries stale content. If the named session is valid, use `--resume <session-name>`. If not, write a summary, not a paste.

---

## 📝 Exam-Style Practice Questions

**Q1.** After a long analysis session, a developer wants to compare two refactoring approaches that both start from that same analysis, without either approach influencing the other. What is the best way?

- A) Explore both in the same conversation, one after the other.
- B) Start two new sessions and redo the analysis in each.
- C) Resume the session and ask Claude to forget Approach A before starting B.
- D) Use `fork_session` to create two independent branches from the analysis baseline.

**Q2.** A developer resumes yesterday's session. Two files were edited overnight; everything else is the same. What is the best approach?

- A) Resume with `--resume <session-name>` and tell the session which two files changed so it re-reads them.
- B) Resume and say nothing. Claude will notice the differences.
- C) Delete the session and start without any summary.
- D) Fork the session and use the fork as is.

**Q3.** After merging forty upstream commits, a developer wants to continue an old investigation. Most earlier tool results no longer match the code. What is best?

- A) Resume and rely on the earlier tool results.
- B) Start a new session seeded with a structured summary of verified findings and the changes, plus the new task.
- C) Resume and ask Claude to guess which results are stale.
- D) Fork the old session and continue in the fork.

### Answers

**Q1: D.**
- A) The two approaches contaminate each other in one thread.
- B) Redoing the analysis twice wastes time and may give different baselines.
- C) Asking to "forget" is unreliable, and the earlier text is still in context.

**Q2: A.**
- B) Claude has no way to know that files changed outside the session.
- C) Throwing away the context loses useful work for a two-file change.
- D) A fork carries the same stale reads. It adds nothing for this case.

**Q3: B.**
- A) The old tool results are stale and would mislead the work.
- C) Claude cannot know which results are stale without re-reading everything.
- D) A fork still contains the stale tool results.

---

## ✅ Self-Check Checklist

- [ ] I can say when to use `--resume <session-name>`.
- [ ] I can say when to use `fork_session`.
- [ ] I know to tell a resumed session which files changed.
- [ ] I can explain when to start a new session with a structured summary.
- [ ] I know what makes a good summary: verified facts, decisions, changes, next task.

**Memory hook**: *"Resume if true, fork to compare, summarise if stale."*

---

## 🛠️ Hands-on Workbook Guide

Open [ts_1_7_session_management.ipynb](ts_1_7_session_management.ipynb) (offline, mock client):

1. **Baseline**: create the first session.
2. **Fork**: an independent branch that leaves the parent unchanged.
3. **Resume**: continue a named session and tell it which files changed.
4. **Fresh session with summary**: the recovery pattern for stale results.
5. **Checks**: forks stay isolated, the changed-files notice is added, and summaries carry only verified facts.

Code: [main.py](main.py) (live API, run manually only) · Offline tests: [tests/test_domain1_enhancements.py](../../tests/test_domain1_enhancements.py)

---

## 🧭 Navigation & Study Links
- ⬅️ **Previous Topic**: [TS 1.6: Dynamic Task Decomposition](../ts_1_6_task_decomposition/README.md)
- 📋 **Domain Overview**: [Domain 1: Agentic Architecture & Orchestration](../README.md)
- ➡️ **Next Topic**: [TS 2.1: Tool Interface Design](../../domain2/ts_2_1_tool_interface_design/README.md)
- 🎯 **Scenario Practice**: [Master Scenario Practice](../../scenarios/MASTER_SCENARIO_PRACTICE.md)

In [ ]:
# Step 1: Environment & Offline Setup
import copy
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 1.7 Session Management.")

### 🗃️ Step 2: Session State & Manager Implementation
We implement `SessionState` supporting branching/forking and `SessionManager` supporting summary injection.

In [ ]:
# Step 2: Session Manager Logic
@dataclass
class SessionState:
    session_id: str
    session_name: str
    messages: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def fork(self, new_name: str) -> "SessionState":
        """Branch from current session checkpoint into an isolated thread."""
        return SessionState(
            session_id=f"fork_{uuid.uuid4().hex[:8]}",
            session_name=new_name,
            messages=copy.deepcopy(self.messages),
            metadata={"parent_session_id": self.session_id},
        )


class SessionManager:
    def __init__(self) -> None:
        self.sessions: Dict[str, SessionState] = {}

    def create_session(self, name: str, initial_messages: Optional[List[Dict[str, Any]]] = None) -> SessionState:
        sid = f"sess_{uuid.uuid4().hex[:8]}"
        sess = SessionState(session_id=sid, session_name=name, messages=initial_messages or [])
        self.sessions[name] = sess
        return sess

    def resume(self, name: str, changed_files: Optional[List[str]] = None) -> SessionState:
        """--resume <session-name>: continue the same history and name any changed files."""
        sess = self.sessions[name]
        if changed_files:
            listed = "\n".join(f"- {p}" for p in changed_files)
            sess.messages.append({
                "role": "user",
                "content": "These files changed since your last tool results. "
                           f"Re-read them before relying on earlier output:\n{listed}",
            })
        return sess

    def resume_with_injected_summary(self, name: str, structured_summary: str, new_instruction: str) -> SessionState:
        """
        Stale Context Remedy: Discards bloated history and starts a clean session
        anchored with a structured summary of previous findings.
        """
        clean_messages = [
            {"role": "user", "content": f"### PREVIOUS ARCHITECTURAL CONTEXT\n{structured_summary}"},
            {"role": "assistant", "content": "Context acknowledged. Ready for next instructions."},
            {"role": "user", "content": new_instruction},
        ]
        return self.create_session(f"{name}_resumed", clean_messages)

print(" SessionManager declared with forking and summary injection.")

### 🧪 Step 3: Interactive Simulation
1. Create a base session.
2. Fork the session to test an alternative architecture in `branch_a`.
3. Verify that adding messages to `branch_a` leaves `base` unpolluted.
4. Mitigate stale context by resuming with an injected summary.
5. Resume a named session and tell it which files changed.

In [ ]:
# Step 3: Simulation
mgr = SessionManager()

# 1. Base session
base = mgr.create_session("base", [{"role": "user", "content": "Design database schema."}])
print(f"Created base session: {base.session_name} with {len(base.messages)} message(s)")

# 2. Fork into branch_a
branch_a = base.fork("branch_a")
branch_a.messages.append({"role": "user", "content": "Explore NoSQL option (MongoDB)"})
print(f"Forked {branch_a.session_name} with {len(branch_a.messages)} messages")
print(f"Base session message count remains: {len(base.messages)}")

# 3. Resume with injected summary
clean = mgr.resume_with_injected_summary(
    "base",
    structured_summary="Agreed schema: PostgreSQL with orders and items tables.",
    new_instruction="Add index on customer_id.",
)
print(f"\nClean resumed session created: {clean.session_name}")
print("Clean messages count:", len(clean.messages))

# 4. Resume a named session and tell it which files changed
mgr.create_session("work", [{"role": "user", "content": "Review db/conn.py"}])
resumed = mgr.resume("work", changed_files=["db/conn.py"])
print("Resume notice:", resumed.messages[-1]["content"])

### 🔬 Step 4: Verification Assertions
We assert that:
1. Forked session preserves parent history at fork time.
2. Mutating the child session does not affect the parent.
3. Resume with summary compresses context and clears stale messages.
4. Resuming with changed files adds a notice that names them.

In [ ]:
# Step 4: Verification Checks
assert branch_a.metadata["parent_session_id"] == base.session_id
assert len(branch_a.messages) == 2
assert len(base.messages) == 1, "Parent session must not be polluted by child fork!"

# Verify summary injection
assert "PREVIOUS ARCHITECTURAL CONTEXT" in clean.messages[0]["content"]
assert "Add index on customer_id" in clean.messages[2]["content"]

assert "db/conn.py" in resumed.messages[-1]["content"]

print(" All verification assertions passed! Session management verified.")